# 01 — Data Ingestion and Parsing

**Purpose:** Demonstrate source-agnostic ingestion and validate parsed Iranian Labor Law records.

Business logic lives in `laborlaw_rag.data`; this notebook only configures and calls the public API.


## 1. Setup


In [ ]:
from laborlaw_rag.config import Settings
from laborlaw_rag.data import (
    LaborLawParser,
    RawSource,
    WebSource,
    fetch_source,
    parse_source,
    save_records,
)

settings = Settings.from_env()

# Keep network access and artifact writes explicit.
REFRESH_SOURCE = False
WRITE_RECORDS = False
LAW_TITLE = "قانون کار"

## 2. Read the Source

Set `REFRESH_SOURCE` to `True` to download the configured website. The cached HTML keeps the default run local and reproducible. A `FileSource` can supply local HTML, text, or PDF without changing later cells.


In [ ]:
if REFRESH_SOURCE or not settings.raw_html_path.exists():
    raw_source = fetch_source(
        WebSource(settings.source_url, title=LAW_TITLE),
        settings.raw_html_path,
    )
else:
    raw_source = RawSource(
        content=settings.raw_html_path.read_text(encoding="utf-8"),
        uri=settings.source_url,
        format="html",
        title=LAW_TITLE,
    )

{"source": raw_source.uri, "format": raw_source.format, "characters": len(raw_source.content)}

## 3. Parse and Validate


In [ ]:
records = parse_source(raw_source)
validation_report = LaborLawParser.validate(records)

assert validation_report["status"] == "PASSED"
validation_report

## 4. Inspect Records


In [ ]:
[
    {
        "record_id": record["record_id"],
        "type": record["type"],
        "article_reference": record.get("article_reference"),
        "text_preview": record["text"][:180],
    }
    for record in records[:3]
]

## 5. Persist the Artifact

Set `WRITE_RECORDS` to `True` only when intentionally refreshing the processed dataset.


In [ ]:
if WRITE_RECORDS:
    save_records(records, settings.records_path, raw_source)

{"write_enabled": WRITE_RECORDS, "records_path": str(settings.records_path)}